# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, which is designed for FAIR AI data in the Croissant schema.

### Dataset Source
The dataset is described via a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
First, we'll load the dataset metadata using the Croissant schema URL with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n{metadata.description}")

## 2. Data Overview
Let's review the available record sets, their fields, and their corresponding `@id`s defined in the Croissant schema.

In [ ]:
# List all record sets and their fields by @id
# - Record sets are the logical tables within the Croissant schema
pprint.pprint([{'@id': rs['@id'], '@type': rs.get('@type'), 'field_ids': [f['@id'] for f in rs.get('field', [])]} for rs in metadata.to_json().get('recordSet', [])])

# If no record sets are found under 'recordSet', try 'record_set', otherwise inform the user.
if not metadata.to_json().get('recordSet', []):
    print("No record sets found in the metadata under 'recordSet'.")
else:
    print(f"Number of record sets: {len(metadata.to_json()['recordSet'])}")

### If there are no record sets found
The metadata returned doesn't directly contain record sets in `recordSet` field. However, for Croissant datasets exported from Sen Science, the available record sets and their IDs are often named with the pattern `'cr:RecordSet/...`'.

Let's enumerate all available record sets from the dataset object using Python code:

In [ ]:
# Explore available record sets using mlcroissant Dataset method.
available_record_sets = dataset.record_sets
print("Available record sets (@id):")
for rs in available_record_sets:
    print(f"- {rs}")

# Let's preview one example record from each record set
for rs in available_record_sets:
    print(f"\nFirst record in record set {rs}:")
    try:
        first = next(dataset.records(record_set=rs))
        pprint.pprint(first)
    except StopIteration:
        print("No records in this record set.")

## 3. Data Extraction
Load data from the main record set(s) into a DataFrame for analysis. All references to record sets, fields, and columns use their Croissant `@id` fields.

**Note:** If there is more than one record set, we'll load each one into a dictionary of DataFrames.

In [ ]:
# Collect all record set IDs
record_sets = dataset.record_sets
dataframes = {}

# Load all records as DataFrames
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    # Only create a DataFrame if there is data
    if records:
        dataframes[record_set] = pd.DataFrame(records)
    else:
        print(f"Warning: Record set {record_set} has no records.")

if dataframes:
    # For demonstration, pick the first record set for further analysis
    main_record_set = list(dataframes.keys())[0]
    print(f"Columns in record set {main_record_set}:\n", dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())
else:
    print("No records available in any record set.")

## 4. Exploratory Data Analysis (EDA)
Let's examine and process a *numeric* field (column) in the main record set DataFrame. We'll filter records, normalize a numeric column, and optionally group the data if a relevant field is available.

### 1. Select a Numeric Field
We'll detect numeric fields automatically and pick one for demonstration. All referenced columns use column names as in the DataFrame which correspond to their `@id` in the schema.

In [ ]:
import numpy as np

# Pick the main DataFrame
df = dataframes[main_record_set]

# Detect numeric fields (int or float columns)
numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_fields:
    print("No numeric fields found in the main record set.")
else:
    # We'll choose the first as the primary field for demo
    numeric_field = numeric_fields[0]
    print(f"Selected numeric field for analysis: {numeric_field}")

    # Set an arbitrary threshold (e.g., median)
    threshold = df[numeric_field].median()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold} (median):")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Optionally, group by the first non-numeric field
    group_fields = [c for c in df.columns if c != numeric_field and df[c].dtype == object]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized values, and optionally the grouped result if a group field was identified.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_fields:
    print("No numeric field available for visualization.")
else:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    # If normalized column exists
    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30)
        plt.title(f'Normalized distribution of {numeric_field}')
        plt.xlabel(f'{numeric_field}_normalized')
        plt.show()

    # If grouped_df exists
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 6))
        sns.barplot(data=grouped_df, x=grouped_df.columns[0], y=grouped_df.columns[1])
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
We demonstrated how to load, explore, and analyze the FAIR^2 dataset using `mlcroissant` via its Croissant schema, referencing all entity and field IDs by their `@id` as recommended. Further analysis can include deeper exploration, modeling, or combining with other datasets as needed for downstream data science tasks.